# Week 2, Model 4: The Cluster Model

**Grouping teams by their style, then predicting from how those groups clash.**

*Author: The Genius Project Year 3*

---

### What you will build
- Groups (clusters) of teams that play in a similar way.
- A prediction built from how each group has beaten the others in the past.
- Win chances and a predicted scoreline for each upcoming fixture.

### The big idea, in one breath
Clustering finds teams that look alike, without being told the answer. Once teams are grouped, we check history to see how one group tends to do against another, and predict from that.

> This notebook uses a realistic teaching dataset. The numbers are believable and
> the patterns are real, but they are not official statistics. The goal is to learn
> how the models think, not to run a betting shop.

**Football note:** we call the sport *football* (some countries call it soccer).
Every stat is explained in plain words below, so you do not need to be a fan to follow along.

### The stats we use, in plain words

| Column | What it means |
| --- | --- |
| `team` | The national team, for example Brazil or Japan. |
| `confederation` | The region a team belongs to. UEFA is Europe, CONMEBOL is South America, CAF is Africa, AFC is Asia, CONCACAF is North and Central America. |
| `fifa_ranking` | The team's position on the official world ladder. 1 is the best. A smaller number is better. |
| `fifa_points` | The score behind the ranking. More points means a stronger team. |
| `elo_rating` | Another strength score, borrowed from chess. Higher is stronger. |
| `world_cup_titles` | How many times the team has won the World Cup. |
| `avg_goals_scored` | Goals the team scores in a typical match. This is their attack. Higher is better. |
| `avg_goals_conceded` | Goals the team lets in during a typical match. This is their defence. Lower is better. |
| `possession_pct` | The share of the match, out of 100, that the team keeps the ball. More possession usually means more control. |
| `pass_accuracy_pct` | Out of every 100 passes, how many reach a team mate. |
| `shots_per_game` | How many attempts at goal the team takes in a match. |
| `shots_on_target_per_game` | Shots that were heading into the net, whether or not they scored. |
| `big_chances_per_game` | Clear chances to score in a match, the kind a player is expected to finish. |
| `clean_sheet_pct` | Out of 100 matches, how many the team finishes without letting in a single goal. |
| `win_rate_last10` | The share of the last 10 matches the team won, from 0 to 1. 0.7 means they won 7 of 10. |
| `form_points_last5` | Points from the last 5 matches. A win is 3 points, a draw is 1, a loss is 0. The most is 15. |
| `squad_avg_age` | The average age of the players in the squad. |
| `squad_value_million_eur` | The total transfer value of the squad, in millions of euros. A rough measure of talent. |
| `top_league_player_share` | The share of players, from 0 to 1, who play for clubs in the strongest leagues. |
| `set_piece_goal_pct` | Out of 100 goals, how many come from free kicks and corners rather than open play. |
| `key_player_injuries` | How many important players are currently injured. |
| `is_2026_host` | 1 if the team is hosting the 2026 World Cup (USA, Canada, Mexico), otherwise 0. |

You do not have to memorise these. Glance back whenever a column name shows up.

### Step 1: Load the data

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the three tables.
teams = pd.read_csv("data/teams.csv")
matches = pd.read_csv("data/matches.csv")
upcoming = pd.read_csv("data/upcoming_matches.csv")

# 2. The numbers we will feed the model for each team.
#    Think of these as the "stats card" for a team.
TEAM_STATS = [
    "fifa_points", "elo_rating", "avg_goals_scored", "avg_goals_conceded",
    "possession_pct", "pass_accuracy_pct", "shots_on_target_per_game",
    "big_chances_per_game", "clean_sheet_pct", "win_rate_last10",
    "form_points_last5", "squad_value_million_eur", "top_league_player_share",
    "key_player_injuries",
]

# 3. A helper that takes a match (team_a vs team_b) and returns one flat row
#    of numbers: team A's stats, team B's stats, and the gap between them.
stats_by_team = teams.set_index("team")

def build_features(team_a, team_b):
    a = stats_by_team.loc[team_a, TEAM_STATS]
    b = stats_by_team.loc[team_b, TEAM_STATS]
    row = {}
    for stat in TEAM_STATS:
        row[f"{stat}_a"] = a[stat]      # team A value
        row[f"{stat}_b"] = b[stat]      # team B value
        row[f"{stat}_diff"] = a[stat] - b[stat]   # the gap (A minus B)
    return row

# 4. Turn every historical match into a row of features.
feature_rows = [build_features(r.team_a, r.team_b) for r in matches.itertuples()]
X_full = pd.DataFrame(feature_rows)

# 5. For the model we keep the GAP columns (team A value minus team B value).
#    These are the easiest to read: a positive weight on a gap means
#    "when team A has more of this than team B, team A is more likely to win."
DIFF_COLS = [c for c in X_full.columns if c.endswith("_diff")]
X = X_full[DIFF_COLS]

# 5. The answer we want the model to learn (the "target").
#    0 = team A won, 1 = draw, 2 = team B won.
def result_code(row):
    if row.winner == row.team_a:
        return 0
    if row.winner == "Draw":
        return 1
    return 2

y = matches.apply(result_code, axis=1)
LABELS = {0: "Team A win", 1: "Draw", 2: "Team B win"}

print("Teams loaded:", len(teams))


### Step 2: The football maths for expected scores

In [ ]:
from math import exp, factorial

def poisson(k, mean):
    "Chance of exactly k goals when a team averages `mean` goals."
    return (mean ** k) * exp(-mean) / factorial(k)

def expected_scoreline(team_a, team_b, max_goals=8):
    """Predict goals for each team and the chance of each result.

    We blend a team's attack with the other team's defence. If A usually
    scores 2.0 and B usually lets in 1.4, A's expected goals sit between them.
    """
    a = stats_by_team.loc[team_a]
    b = stats_by_team.loc[team_b]
    xg_a = 0.5 * (a["avg_goals_scored"] + b["avg_goals_conceded"])
    xg_b = 0.5 * (b["avg_goals_scored"] + a["avg_goals_conceded"])

    p_a = p_draw = p_b = 0.0
    for ga in range(max_goals + 1):
        for gb in range(max_goals + 1):
            p = poisson(ga, xg_a) * poisson(gb, xg_b)
            if ga > gb:
                p_a += p
            elif ga == gb:
                p_draw += p
            else:
                p_b += p
    return {
        "expected_goals_a": round(xg_a, 2),
        "expected_goals_b": round(xg_b, 2),
        "predicted_score": f"{round(xg_a)} - {round(xg_b)}",
        "score_win_prob_a": round(p_a, 3),
        "score_draw_prob": round(p_draw, 3),
        "score_win_prob_b": round(p_b, 3),
    }


### Step 3: Find the groups of teams

We use K-Means, which sorts teams into a chosen number of groups so that teams in the same group have similar stats. We ask for 4 groups, roughly: powerhouses, strong sides, solid teams, and underdogs. K-Means does not know these names, it finds the split on its own.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

STYLE_STATS = ["avg_goals_scored", "avg_goals_conceded", "possession_pct",
               "win_rate_last10", "squad_value_million_eur", "fifa_points"]

scaler = StandardScaler()
team_matrix = scaler.fit_transform(teams[STYLE_STATS])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
teams["cluster"] = kmeans.fit_predict(team_matrix)

# Rank the clusters from strongest to weakest by average FIFA points, so the
# group numbers are easy to read.
order = teams.groupby("cluster")["fifa_points"].mean().sort_values(ascending=False)
tier_name = {c: f"Tier {i+1}" for i, c in enumerate(order.index)}
teams["tier"] = teams["cluster"].map(tier_name)
stats_by_team = teams.set_index("team")   # refresh with the new columns

print("Which teams landed in each tier:")
for tier in sorted(teams["tier"].unique()):
    members = ", ".join(teams.loc[teams["tier"] == tier, "team"])
    print(f"  {tier}: {members}")

### Step 4: How do the tiers do against each other?

For every past match we look up the two teams' tiers and record who won. That gives us a win rate for any tier-versus-tier clash.

In [ ]:
tier_of = teams.set_index("team")["tier"].to_dict()

records = []
for r in matches.itertuples():
    if r.winner == "Draw":
        outcome = "draw"
    elif r.winner == r.team_a:
        outcome = "a"
    else:
        outcome = "b"
    records.append((tier_of[r.team_a], tier_of[r.team_b], outcome))

hist = pd.DataFrame(records, columns=["tier_a", "tier_b", "outcome"])

def matchup_probs(tier_a, tier_b):
    "Win / draw / loss chances for tier_a against tier_b, from history."
    games = hist[(hist.tier_a == tier_a) & (hist.tier_b == tier_b)]
    if len(games) < 5:   # not enough head to head, fall back to a fair guess
        return 0.4, 0.25, 0.35
    p_a = (games.outcome == "a").mean()
    p_d = (games.outcome == "draw").mean()
    p_b = (games.outcome == "b").mean()
    return p_a, p_d, p_b

print("Example: Tier 1 vs Tier 4 ->", matchup_probs("Tier 1", "Tier 4"))

### Step 5: Predict the upcoming World Cup matches

We combine two views: the tier history for win chances, and the attack/defence maths for the scoreline.

In [ ]:
for m in upcoming.itertuples():
    ta, tb = tier_of[m.team_a], tier_of[m.team_b]
    p_a, p_d, p_b = matchup_probs(ta, tb)
    score = expected_scoreline(m.team_a, m.team_b)
    print(f"{m.team_a} ({ta}) vs {m.team_b} ({tb})   ({m.stage})")
    print(f"   {m.team_a} win %      {round(p_a*100,1)}")
    print(f"   draw %              {round(p_d*100,1)}")
    print(f"   {m.team_b} win %      {round(p_b*100,1)}")
    print(f"   predicted score     {score['predicted_score']}")
    print(f"   expected goals      {score['expected_goals_a']} - {score['expected_goals_b']}")
    print()

### Your turn

1. Change `n_clusters=4` to `n_clusters=3` or `6`. How do the tiers change?
2. Add `pass_accuracy_pct` to `STYLE_STATS` and re-run. Does any team move tier?
3. Clustering does not use the match results to form groups. Explain, in one sentence, why that makes it different from the tree and the neural network.

## Submit your prediction

You have trained a model and predicted real fixtures. Now enter your answers
in the Week 2 quiz and prediction page:

**https://beagenius.org/tgp-2026teens-week2/06-quiz-and-predict.html**

Bring three things with you for a match of your choice:

1. The win probability for each team.
2. The predicted scoreline (the expected goals rounded to whole numbers).
3. One sentence on which stat mattered most, in your own words.

Good luck, and trust your numbers.
